In [1]:
include("../RayTracing.jl")

Main.RayTracing

In [10]:
function union_bounding_spheres(sphere1::RayTracing.Sphere, sphere2::RayTracing.Sphere)::RayTracing.Sphere
    center1 = sphere1.core.world_to_object(RayTracing.Pnt3(0,0,0))
    center2 = sphere2.core.world_to_object(RayTracing.Pnt3(0,0,0))

    # Get the vector between the centers
    center_vector = center2 - center1
    center_distance = RayTracing.norm(center_vector)
    
    # If one sphere contains the other, return the larger one
    if center_distance + sphere2.radius <= sphere1.radius
        # sphere1 completely contains sphere2
        return sphere1
    elseif center_distance + sphere1.radius <= sphere2.radius
        # sphere2 completely contains sphere1
        return sphere2
    end
    
    # Otherwise, create a new sphere that encloses both
    # The new center is along the line between the centers
    # weighted by the radii
    new_center = center1 + center_vector * 0.5
    
    # The new radius must reach the furthest point of either sphere
    new_radius = (center_distance + sphere1.radius + sphere2.radius) * 0.5
    
    return RayTracing.Sphere(new_center, new_radius)
end

union_bounding_spheres (generic function with 1 method)

In [71]:
################
### ABSTRACT ###
################

# Base abstract type for all SDF components
abstract type ImplicitSurface end

# Primitive SDF shapes inherit from this
abstract type SDFPrimitive <: ImplicitSurface end

# Operations (union, intersection, etc.) inherit from this
abstract type SDFOperation <: ImplicitSurface end

##################
### OPERATIONS ###
##################

# Define specific operation types
struct SDFUnion <: SDFOperation
    k::Float64  # Smoothing parameter
    left::ImplicitSurface
    right::ImplicitSurface
    bounding_sphere::RayTracing.Sphere

    function SDFUnion(k::Float64, left::ImplicitSurface, right::ImplicitSurface)
        return new(k, left, right, union_bounding_spheres(left.bounding_sphere, right.bounding_sphere))
    end
end

##################
### PRIMITIVES ###
##################

struct SDFSphere <: SDFPrimitive
    radius::Float64
    core::RayTracing.ShapeCore
    bounding_sphere::RayTracing.Sphere
end

struct SDFBox <: SDFPrimitive
    half_extents::RayTracing.Pnt3  # half-width, half-height, half-depth
    core::RayTracing.ShapeCore
    bounding_sphere::RayTracing.Sphere
end

###################
### EVALUTATION ###
###################

# Primitive evaluations
function evaluate(shape::SDFSphere, p::RayTracing.Pnt3)::Float64
    # Transform point to object space
    local_p = shape.core.world_to_object(p)
    # Sphere SDF: length(p) - radius
    return RayTracing.norm(local_p) - shape.radius
end

function evaluate(shape::SDFBox, p::RayTracing.Pnt3)::Float64
    # Transform point to object space
    local_p = shape.core.world_to_object(p)
    # Box SDF implementation
    q = abs.(local_p) .- shape.half_extents
    return RayTracing.norm(max.(q, 0.0)) + min(maximum(q), 0.0)
end

# Operation evaluations
function evaluate(op::SDFUnion, p::RayTracing.Pnt3)::Float64
    a = evaluate(op.left, p) 
    b = evaluate(op.right, p)
    
    # Smooth union formula
    if op.k > 0
        h = clamp(0.5 + 0.5 * (b - a) / op.k, 0.0, 1.0)
        return min(b, a, h) - op.k * h * (1.0 - h)
    else
        # Regular union
        return min(a, b)
    end
end

function evaluate(op::SDFIntersection, p::RayTracing.Pnt3)::Float64
    a = evaluate(op.left, p)
    b = evaluate(op.right, p)
    
    # Smooth intersection formula
    if op.k > 0
        h = clamp(0.5 - 0.5 * (b - a) / op.k, 0.0, 1.0)
        return mix(b, a, h) + op.k * h * (1.0 - h)
    else
        # Regular intersection
        return max(a, b)
    end
end

#################
### INTERFACE ###
#################

# Main evaluation function - dispatches to specialized methods
function f(element::ImplicitSurface, p::RayTracing.Pnt3)::Float64
    return evaluate(element, p)
end

# Compatibility with ray evaluation
function f(element::ImplicitSurface, t::Float64, r::RayTracing.AbstractRay)::Float64
    return f(element, RayTracing.at(r, t))
end

# Create two primitive shapes
sphere = SDFSphere(
    1.0, 
    RayTracing.ShapeCore(), 
    RayTracing.Sphere(RayTracing.Pnt3(0,0,0), 1.0 * 1.1)
)
box = SDFBox(RayTracing.Pnt3(1.0, 1.0, 1.0), RayTracing.ShapeCore(), RayTracing.Sphere(RayTracing.Pnt3(0,0,0), 3.0 * 1.1))

# Create a union of the two shapes
union_shape = SDFUnion(0.0, sphere, box)

# Evaluate the SDF at a point
distance_s = f(sphere, RayTracing.Pnt3(10.0, 7.0, 5.0))
distance_b = f(box, RayTracing.Pnt3(10.0, 7.0, 5.0))

11.532562594670797

# why no smooth union working?

In [48]:
o = RayTracing.Pnt3(10, 7, 5)

r = RayTracing.Ray(
    o,
    RayTracing.normalize(RayTracing.Vec3(RayTracing.Pnt3(0,0,0) - o)),
    0.0,
    typemax(Float64)
)

[10.0, 7.0, 5.0], [-0.7580980435789033, -0.5306686305052324, -0.37904902178945166], 0.0, Inf, false

In [49]:
function intersect_t(s::Union{SDFUnion, SDFPrimitive}, r::RayTracing.AbstractRay)::Float64
    # set up anonymous function for solver
    tmp_solve = (x -> f(s, x, r))

    # intersect bounding sphere
    check, t0, t1 = RayTracing.intersect_simple(s.bounding_sphere, r)

    # doesn't intersect sphere, NEXT
    if !check
        return -1.0
    else
        @info "Implicit Surface Bounding Sphere Intersection: $(RayTracing.at(r, t0)), $(t0), $(t1)"
    end

    # TODO some checks t0 & t1 aren't negative?

    # skipping solve if start point is zero
    if tmp_solve(t0) == 0.0
        solutions = [t0]
    elseif tmp_solve(t1) == 0.0
        solutions = [t1]
    else
        @info "Entering Solve: $r"
        solutions = RayTracing.find_zeros(tmp_solve, t0, t1) # HACKY
        @info "Exiting Solve: $solutions"
    end

    @info "ImplicitSurfaceIntersectionTest: ray: $(r), solutions: $(solutions), bounding_sphere bounds: ($(t0/1.1), $(t1*1.1))"

    if length(solutions) == 0
        @info "Length of solutions is zero"
        return -1.0
    end

    # find intersection time
    _, idx = findmin(abs.(solutions))
    t = solutions[idx]

    if t > r.tMax
        @info "t out of bounds for the ray"
        return -1.0
    end

    return t
end

intersect_t (generic function with 1 method)

In [59]:
t = intersect_t(sphere, r)

┌ Info: Implicit Surface Bounding Sphere Intersection: [0.8339078479367963, 0.583735493555757, 0.41695392396839814], 12.090905958272916, 14.290905958272923
└ @ Main /Users/johnmyslinski/Documents/PBRJ/src/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X35sZmlsZQ==.jl:12
┌ Info: Entering Solve: [10.0, 7.0, 5.0], [-0.7580980435789033, -0.5306686305052324, -0.37904902178945166], 0.0, Inf, false
└ @ Main /Users/johnmyslinski/Documents/PBRJ/src/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X35sZmlsZQ==.jl:23
┌ Info: Exiting Solve: [12.19090595827292, 14.19090595827292]
└ @ Main /Users/johnmyslinski/Documents/PBRJ/src/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X35sZmlsZQ==.jl:25
┌ Info: ImplicitSurfaceIntersectionTest: ray: [10.0, 7.0, 5.0], [-0.7580980435789033, -0.5306686305052324, -0.37904902178945166], 0.0, Inf, false, solutions: [12.19090595827292, 14.19090595827292], bounding_sphere bounds: (10.991732689339013, 15.719996554100216)
└ @ Main /U

12.19090595827292

In [60]:
RayTracing.at(r, t)

3-element Main.RayTracing.Pnt3 with indices SOneTo(3):
 0.7580980435789044
 0.5306686305052324
 0.3790490217894522

In [67]:
distance = f(sphere, RayTracing.at(r, t))

8.881784197001252e-16